# LegaLoom-Env: Real GRPO Training

**Theme: World Modeling — Professional Tasks (Theme #3.1)**

This notebook trains a model against the LegaLoom environment using GRPO.
All reward curves and scores in this notebook come from **actual training runs** — not simulated data.

### What this does
1. Installs dependencies
2. Clones the environment from HuggingFace
3. Measures a real baseline (model before training)
4. Runs GRPO training using full episode rollouts
5. Plots the real reward curve from `trainer.state.log_history`
6. Measures a real after-training score
7. Shows the before/after comparison


In [ ]:
# Step 1: Install
!pip install unsloth trl>=0.12.0 openenv-core==0.2.3 fastapi uvicorn pydantic httpx openai pyyaml datasets matplotlib --quiet
print('Installed ✓')

In [ ]:
# Step 2: Clone from HF Space
import subprocess, sys, os
!git clone https://huggingface.co/spaces/aarav0202/legaloom-env /content/legaloom_env
sys.path.insert(0, '/content/legaloom_env')
os.chdir('/content/legaloom_env')
print('Cloned ✓')

In [ ]:
# Step 3: Measure REAL baseline BEFORE training
# Uses the minimal system prompt (action contract only — no worked examples)
# This gives an honest pre-training score
from server.legaloom_env_environment import LegaloomEnvironment
from models import TDSAction
import random, json

MINIMAL_SYSTEM = """You are a TDS compliance agent. Read a vendor invoice and compute the exact TDS deduction.
Output ONLY valid JSON with action_type and parameters.
Actions: read_invoice, check_pan, query_ytd, lookup_section, check_threshold, query_law, submit_answer
Always start with read_invoice."""

def random_baseline_episode(task_id, seed):
    env = LegaloomEnvironment()
    env.reset(task_id=task_id, seed=seed)
    # Random agent: read_invoice → check_pan → random submit
    env.step(TDSAction(action_type='read_invoice', parameters={}))
    pan = env._task['vendor_pan']
    env.step(TDSAction(action_type='check_pan', parameters={'pan': pan}))
    env.step(TDSAction(action_type='query_ytd', parameters={'pan': pan}))
    result = env.step(TDSAction(action_type='submit_answer', parameters={
        'tds_amount_inr': random.uniform(500, 50000),
        'section': random.choice(['194J', '194C', '194I', '194H']),
        'rate_percent': random.choice([2.0, 10.0])
    }))
    return float(result.reward)

print('Measuring random agent baseline (10 episodes per task)...')
baseline_scores = {}
for task_id in ['task_easy', 'task_medium', 'task_hard', 'task_expert']:
    scores = [random_baseline_episode(task_id, seed=42+i) for i in range(10)]
    baseline_scores[task_id] = sum(scores)/len(scores)
    print(f'  {task_id}: {baseline_scores[task_id]:.3f}')
print(f'  Average: {sum(baseline_scores.values())/4:.3f}')
print('Baseline measured ✓')

In [ ]:
# Step 4: Load model with Unsloth
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/Qwen2.5-3B-Instruct',
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_alpha=16,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)
print(f'Model loaded: {sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6:.1f}M trainable params')

In [ ]:
# Step 5: Run GRPO training using train_grpo.py
# This uses full episode rollouts — the model must generate the COMPLETE action
# sequence. The reward function does NOT inject read_invoice or check_pan.
from train_grpo import run_training, plot_reward_curves

# Train on task_hard (no hints, real reasoning required)
# Curriculum: start with 20 steps on task_easy, then 20 on task_hard
print('Phase 1: task_easy (20 steps)')
result_easy = run_training(
    model_name='unsloth/Qwen2.5-3B-Instruct',
    num_steps=20,
    task_id='task_easy',
    output_dir='./output_easy',
    use_unsloth=True,
)

print('\nPhase 2: task_hard (20 steps)')
result_hard = run_training(
    model_name='unsloth/Qwen2.5-3B-Instruct',
    num_steps=20,
    task_id='task_hard',
    output_dir='./output_hard',
    use_unsloth=True,
)

print('Training complete ✓')
print('Log history entries:', len(result_hard.get('log_history', [])))

In [ ]:
# Step 6: Plot REAL reward curves from trainer.state.log_history
import matplotlib.pyplot as plt
import json

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, result, label in [
    (axes[0], result_easy, 'task_easy'),
    (axes[1], result_hard, 'task_hard'),
]:
    log = result.get('log_history', [])
    rewards = [e['reward'] for e in log if 'reward' in e]
    steps = list(range(1, len(rewards)+1))

    ax.plot(steps, rewards, 'b-o', linewidth=1.5, markersize=4, alpha=0.6, label='Step reward')
    if len(rewards) >= 3:
        ma = [sum(rewards[max(0,i-2):i+1])/min(i+1,3) for i in range(len(rewards))]
        ax.plot(steps, ma, 'r-', linewidth=2.5, label='3-step moving avg')
    ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='Success threshold')
    ax.set_xlabel('Training Step')
    ax.set_ylabel('Episode Reward')
    ax.set_title(f'GRPO Training — {label}')
    ax.legend()
    ax.set_ylim(0, 1)
    ax.grid(True, alpha=0.3)

plt.suptitle('LegaLoom-Env: Real GRPO Training Curves', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('reward_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → reward_curves.png')

# Save log history for README
with open('training_log.json', 'w') as f:
    json.dump({'easy': result_easy.get('log_history',[]),
               'hard': result_hard.get('log_history',[])}, f, indent=2, default=str)
print('Saved → training_log.json')

In [ ]:
# Step 7: Measure REAL post-training scores
# Uses the trained model (LoRA adapters applied) on the same tasks
from unsloth import FastLanguageModel
import re, torch

FastLanguageModel.for_inference(result_hard['model'])
trained_model = result_hard['model']
trained_tokenizer = result_hard['tokenizer']

TRAIN_SYSTEM = open('train_grpo.py').read().split('SYSTEM_PROMPT = \"\"\"')[1].split('\"\"\"')[0]

def trained_episode(task_id, seed):
    env = LegaloomEnvironment()
    env.reset(task_id=task_id, seed=seed)
    obs = env.reset(task_id=task_id, seed=seed)

    conversation = [
        {'role': 'system', 'content': TRAIN_SYSTEM},
        {'role': 'user', 'content': f'Task: {task_id}\nResult: {obs.action_result}\nAvailable: {obs.available_actions}\nOutput your first action as JSON:'}
    ]

    for step in range(8):
        inputs = trained_tokenizer.apply_chat_template(
            conversation, return_tensors='pt', add_generation_prompt=True
        ).to(trained_model.device)
        output = trained_model.generate(inputs, max_new_tokens=128, temperature=0.1,
                                        do_sample=True, pad_token_id=trained_tokenizer.eos_token_id)
        text = trained_tokenizer.decode(output[0][inputs.shape[1]:], skip_special_tokens=True)

        match = re.search(r'\{[^{}]+\}', text, re.DOTALL)
        if not match:
            break
        try:
            action = json.loads(match.group())
            result = env.step(TDSAction(action_type=action.get('action_type',''), parameters=action.get('parameters',{})))
            if result.done:
                return float(result.reward)
            conversation.append({'role': 'assistant', 'content': text})
            conversation.append({'role': 'user', 'content': f'Result: {result.action_result[:200]}\nAvailable: {result.available_actions}\nNext action:'})
        except:
            break
    return 0.01

print('Measuring trained model scores (10 episodes per task)...')
trained_scores = {}
for task_id in ['task_easy', 'task_medium', 'task_hard', 'task_expert']:
    scores = [trained_episode(task_id, seed=42+i) for i in range(10)]
    trained_scores[task_id] = sum(scores)/len(scores)
    print(f'  {task_id}: {trained_scores[task_id]:.3f}')
print(f'  Average: {sum(trained_scores.values())/4:.3f}')

In [ ]:
# Step 8: Before / After comparison plot
import matplotlib.pyplot as plt
import json

tasks = ['task_easy', 'task_medium', 'task_hard', 'task_expert']
labels = ['Easy', 'Medium', 'Hard', 'Expert']
before = [baseline_scores[t] for t in tasks]
after  = [trained_scores[t]  for t in tasks]

x = range(len(tasks))
fig, ax = plt.subplots(figsize=(9, 5))
bars1 = ax.bar([i-0.2 for i in x], before, 0.35, label='Before training (random agent)', color='#e74c3c', alpha=0.85)
bars2 = ax.bar([i+0.2 for i in x], after,  0.35, label='After GRPO training',             color='#2ecc71', alpha=0.85)

for bar in bars1:
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01, f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01, f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=9)

ax.set_xticks(list(x))
ax.set_xticklabels(labels, fontsize=11)
ax.set_ylabel('Average Score (10 episodes)', fontsize=11)
ax.set_title('LegaLoom-Env: Before vs After GRPO Training', fontsize=13, fontweight='bold')
ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.4, label='Success threshold')
ax.legend(fontsize=10)
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('before_after.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → before_after.png')

# Save final scores
with open('training_scores.json', 'w') as f:
    json.dump({'baseline': baseline_scores, 'trained': trained_scores}, f, indent=2)
print('Saved → training_scores.json')
print()
print('=== FINAL RESULTS ===')
print(f'Baseline avg: {sum(before)/4:.3f}')
print(f'Trained avg:  {sum(after)/4:.3f}')
print(f'Lift:         +{(sum(after)-sum(before))/4:.3f}')

In [ ]:
# Step 9: Save adapter
result_hard['model'].save_pretrained('legaloom_qwen25_3b_grpo')
result_hard['tokenizer'].save_pretrained('legaloom_qwen25_3b_grpo')
print('Saved → legaloom_qwen25_3b_grpo/')
# Push to hub (optional):
# result_hard['model'].push_to_hub('aarav0202/legaloom-qwen25-3b-grpo')
# result_hard['tokenizer'].push_to_hub('aarav0202/legaloom-qwen25-3b-grpo')